
#### refunder agent

this notebook creates an agent with tools to suggest refunds for orders

#### Tool & View Registration

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ${CATALOG}.ai;

In [ ]:
%sql
CREATE OR REPLACE FUNCTION ${CATALOG}.ai.get_order_details(oid STRING COMMENT 'order id of the order')
RETURNS TABLE (
  body STRING COMMENT 'Body of the event',
  event_type STRING COMMENT 'The type of event',
  order_id STRING COMMENT 'The order id',
  ts STRING COMMENT 'The timestamp of the event',
  location STRING COMMENT 'the location of the order'
)
COMMENT 'Returns all events associated with the order id (oid)'
RETURN
  SELECT ae.body, ae.event_type, ae.order_id, ae.ts, loc.name as location
  FROM ${CATALOG}.lakeflow.all_events ae
  LEFT JOIN ${CATALOG}.simulator.locations loc ON ae.location_id = loc.location_id
  WHERE ae.order_id = oid;

In [0]:
%sql
CREATE OR REPLACE FUNCTION ${CATALOG}.ai.get_order_delivery_time(oid STRING COMMENT 'order id of the order')
RETURNS TABLE (
  order_id STRING COMMENT 'The order id',
  creation_time TIMESTAMP COMMENT 'The timestamp of the first event for the order',
  delivery_time TIMESTAMP COMMENT 'The timestamp of the last event for the order',
  duration_minutes FLOAT COMMENT 'The total duration from the first to the last event in minutes'
)
COMMENT 'Returns the first event time, last event time, and total duration for a given order id.'
RETURN
  WITH MinMaxTimestamps AS (
    SELECT
      MIN(try_to_timestamp(ts)) as first_event_time,
      MAX(try_to_timestamp(ts)) as last_event_time
    FROM
      ${CATALOG}.lakeflow.all_events
    WHERE
      order_id = oid
  )
  SELECT
    oid as order_id,
    first_event_time AS creation_time,
    last_event_time AS delivery_time,
    CAST(
      try_divide(
        (UNIX_TIMESTAMP(last_event_time) - UNIX_TIMESTAMP(first_event_time)),
        60
      ) AS FLOAT
    ) AS duration_minutes
  FROM
    MinMaxTimestamps;

In [ ]:
%sql
CREATE OR REPLACE VIEW ${CATALOG}.ai.order_delivery_times_per_location_view AS
WITH order_times AS (
  SELECT
    ae.order_id,
    loc.name as location,
    MAX(CASE WHEN ae.event_type = 'order_created' THEN try_to_timestamp(ae.ts) END) AS order_created_time,
    MAX(CASE WHEN ae.event_type = 'delivered' THEN try_to_timestamp(ae.ts) END) AS delivered_time
  FROM
    ${CATALOG}.lakeflow.all_events ae
  LEFT JOIN ${CATALOG}.simulator.locations loc ON ae.location_id = loc.location_id
  WHERE
    try_to_timestamp(ae.ts) >= CURRENT_TIMESTAMP() - INTERVAL 1 DAY
  GROUP BY
    ae.order_id,
    loc.name
),
total_order_times AS (
  SELECT
    order_id,
    location,
    (UNIX_TIMESTAMP(delivered_time) - UNIX_TIMESTAMP(order_created_time)) / 60 AS total_order_time_minutes
  FROM
    order_times
  WHERE
    order_created_time IS NOT NULL
    AND delivered_time IS NOT NULL
)
SELECT
  location,
  PERCENTILE(total_order_time_minutes, 0.50) AS P50,
  PERCENTILE(total_order_time_minutes, 0.75) AS P75,
  PERCENTILE(total_order_time_minutes, 0.99) AS P99
FROM
  total_order_times
GROUP BY
  location

In [0]:
%sql
CREATE OR REPLACE FUNCTION ${CATALOG}.ai.get_location_timings(loc STRING COMMENT 'Location name as a string')
RETURNS TABLE (
  location STRING COMMENT 'Location of the order source',
  P50 FLOAT COMMENT '50th percentile',
  P75 FLOAT COMMENT '75th percentile',
  P99 FLOAT COMMENT '99th percentile'
)
COMMENT 'Returns the 50/75/99th percentile of total delivery times for locations'
RETURN
  SELECT location, P50, P75, P99
  FROM ${CATALOG}.ai.order_delivery_times_per_location_view AS odlt
  WHERE odlt.location = loc;

In [ ]:
%sql
-- USE CATALOG is needed in addition to USE SCHEMA + EXECUTE so the serving
-- endpoint's auto-generated SP can traverse the catalog to reach the UC
-- functions at model-load time.  Normally granted by the root data stage
-- (canonical_data/raw_data), but repeated here so this stage is self-
-- sufficient if run standalone or against a pre-existing catalog.
GRANT USE CATALOG ON CATALOG ${CATALOG} TO `account users`;
GRANT USE SCHEMA ON SCHEMA ${CATALOG}.ai TO `account users`;

In [ ]:
%sql
-- Grant EXECUTE so the serving endpoint SP can call these tools at inference time.
GRANT EXECUTE ON FUNCTION ${CATALOG}.ai.get_order_details        TO `account users`;
GRANT EXECUTE ON FUNCTION ${CATALOG}.ai.get_order_delivery_time  TO `account users`;
GRANT EXECUTE ON FUNCTION ${CATALOG}.ai.get_location_timings     TO `account users`;

#### Model

In [0]:
%pip install -U -qqqq mlflow-skinny[databricks] "langgraph>=0.3.5,<0.4.0" databricks-langchain langchain-openai databricks-agents uv
dbutils.library.restartPython()

In [0]:
CATALOG = dbutils.widgets.get("CATALOG")
LLM_MODEL = dbutils.widgets.get("LLM_MODEL")

# Unity AI Gateway (v2 Beta) endpoint name.  Defined as a job parameter on
# the `all` target only — empty string everywhere else.  When set, the
# agent routes its internal LLM calls through the gateway URL
# (<host>/ai-gateway/mlflow/v1) instead of calling a foundation-model
# serving endpoint directly.  See databricks.yml comment on the param for
# the full rationale + UI setup steps.
try:
    AI_GATEWAY_ENDPOINT_NAME = dbutils.widgets.get("AI_GATEWAY_ENDPOINT_NAME")
except Exception:
    # Widget not declared on this target — equivalent to "no gateway".
    AI_GATEWAY_ENDPOINT_NAME = ""

if AI_GATEWAY_ENDPOINT_NAME:
    print(f"⛩  Routing refund agent LLM calls through AI Gateway: {AI_GATEWAY_ENDPOINT_NAME}")
else:
    print(f"→ Refund agent LLM calls go directly to foundation-model endpoint: {LLM_MODEL}")

In [ ]:
import mlflow

# Create/set dev experiment for development and evaluation traces.
# Use a shared path so the job-runner SP and the deployer both write to the
# same experiment and the runbook can link to a stable URL.  Mirrors the
# pattern in stages/complaint_agent.ipynb so all three pipeline agents
# (refund / complaint / supervisor) share the same dev/prod split.
dev_experiment_name = f"/Shared/{CATALOG}_refund_agent_dev"

# set_experiment creates the experiment if it doesn't exist, or activates it if it does.
dev_experiment = mlflow.set_experiment(dev_experiment_name)
dev_experiment_id = dev_experiment.experiment_id
print(f"✅ Using dev experiment: {dev_experiment_name} (ID: {dev_experiment_id})")

# Track the experiment in uc_state so `databricks bundle run cleanup` deletes it.
import sys
sys.path.append('../utils')
from uc_state import add

experiment_data = {
    "experiment_id": dev_experiment_id,
    "name": dev_experiment_name,
}
add(CATALOG, "experiments", experiment_data)
print(f"✅ Added dev experiment to UC state")

In [0]:
import re
import os
from IPython.core.magic import register_cell_magic

# Records absolute paths of files written by `%%writefilev`, keyed by the
# filename argument.  Cell 17 below imports `from agent import LLM_ENDPOINT_NAME`.
#
# Earlier versions wrote to `os.path.abspath(filename)` (i.e. CWD), but CWD
# on serverless notebooks is usually a `/Workspace/Users/...` path, and the
# workspace-files-as-Python-modules feature is not reliably wired through
# on serverless — the file ends up on disk, `os.path.exists()` returns True,
# but `import agent` still raises `ModuleNotFoundError`.  Pinning the write
# dir to a regular local-disk path (`/local_disk0/tmp/...`, falling back to
# `/tmp/...`) sidesteps the workspace-files importer entirely.
_WRITEFILEV_ABS_PATHS = {}

_WRITEFILEV_DIR = "/local_disk0/tmp/caspers_writefilev"
if not os.path.isdir("/local_disk0"):
    _WRITEFILEV_DIR = "/tmp/caspers_writefilev"
os.makedirs(_WRITEFILEV_DIR, exist_ok=True)

@register_cell_magic
def writefilev(line, cell):
    """
    %%writefilev file.py
    Allows {{var}} substitutions while leaving normal {} intact.

    Writes to a stable local-disk path (NOT CWD) so subsequent
    `from <module> import ...` always succeeds, even on serverless
    where CWD is a /Workspace path.
    """
    filename = line.strip()

    def replacer(match):
        expr = match.group(1)
        return str(eval(expr, globals(), locals()))

    content = re.sub(r"\{\{(.*?)\}\}", replacer, cell)

    abs_path = os.path.join(_WRITEFILEV_DIR, filename)
    with open(abs_path, "w") as f:
        f.write(content)
    _WRITEFILEV_ABS_PATHS[filename] = abs_path
    print(f"Wrote file with substitutions: {abs_path}")

In [0]:
%%writefilev agent.py
from typing import Any, Generator, Literal, Optional, Sequence, Union

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    VectorSearchRetrieverTool,
)
from langchain_core.tools import tool
from unitycatalog.ai.core.base import get_uc_function_client
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
try:
    from langgraph.graph.graph import CompiledGraph
    from langgraph.graph.state import CompiledStateGraph
except ImportError:  # langgraph >=0.4 restructured these internal modules
    from typing import Any
    CompiledGraph = Any
    CompiledStateGraph = Any
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentChunk,
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)

import json as _json
import uuid as _uuid
from pydantic import BaseModel, ValidationError

mlflow.langchain.autolog()

# Catalog and AI-Gateway endpoint name are substituted at notebook write
# time (via `%%writefilev {{...}}`) so the agent module is self-contained
# and does not depend on widgets at request time.
CATALOG = "{{CATALOG}}"

# When non-empty, the agent routes every LLM call through the Unity AI
# Gateway (v2 Beta) URL `<host>/ai-gateway/mlflow/v1` so the gateway's
# guardrails / inference table / usage tracking / rate limits apply
# centrally.  When empty, the agent calls the foundation-model serving
# endpoint named by LLM_ENDPOINT_NAME directly via ChatDatabricks
# (legacy behaviour, used by the `default` and `complaints` targets).
AI_GATEWAY_ENDPOINT_NAME = "{{AI_GATEWAY_ENDPOINT_NAME}}"

# Lazy UC function client — constructed on first tool call, NOT at
# module import / model load time.  This is the same pattern used by
# complaint_agent.py.  It matters because the serving endpoint's
# auto-created service principal is added to `account users` (which
# holds USE CATALOG / USE SCHEMA / EXECUTE) with a multi-minute
# propagation delay; if we introspect or execute UC functions at model
# LOAD time (the way the old UCFunctionToolkit code did), the first
# deploy fails before propagation completes.  Deferring all UC access to
# request time lets model load complete unconditionally, and by the time
# real traffic arrives the SP has propagated.
_uc_client = None


def _client():
    global _uc_client
    if _uc_client is None:
        _uc_client = get_uc_function_client()
    return _uc_client


class RefundDecision(BaseModel):
    refund_usd: float = 0.0
    refund_class: Literal["none", "partial", "full"] = "none"
    reason: str = ""


############################################
# Define your LLM endpoint and system prompt
############################################
# `LLM_ENDPOINT_NAME` is the name we expose for `resources=[...]` and for
# logging.  When the AI Gateway is wired up, it points at the gateway
# endpoint (so the deploy-time SP gets CAN_QUERY auto-granted on the
# gateway endpoint).  When not, it points at the foundation-model
# endpoint as before.
LLM_ENDPOINT_NAME = AI_GATEWAY_ENDPOINT_NAME or "{{LLM_MODEL}}"

if AI_GATEWAY_ENDPOINT_NAME:
    # Route through Unity AI Gateway (v2 Beta).  The gateway URL is
    # OpenAI-compatible at `<host>/ai-gateway/mlflow/v1` and expects the
    # gateway-endpoint-name as the `model` field in the request body.
    #
    # Auth: in Model Serving the runtime authenticates as the deployed
    # endpoint's service principal via OAuth M2M.  In M2M mode there is
    # NO static token — `WorkspaceClient().config.token` returns None,
    # which then crashes the OpenAI SDK with
    #     OpenAIError: Missing credentials. Please pass an api_key ...
    # at agent construction time.
    #
    # The portable fix is `config.authenticate()`, which returns the
    # auth headers that the SDK would attach to a request right now —
    # `{"Authorization": "Bearer <token>"}` — regardless of whether the
    # underlying auth provider is PAT, OAuth U2M, or OAuth M2M.  We
    # strip the bearer prefix and hand the raw token to ChatOpenAI as
    # `api_key`.
    #
    # Token rotation — this used to be a caveat (the bearer was extracted
    # once at module load and baked into ChatOpenAI, so warm endpoints
    # surfaced HTTP 400 BAD_REQUEST / "Invalid Token" from the gateway
    # after the original OAuth M2M token's ~1h window closed; the agent
    # had to cold-reload to recover).  Fixed by mitigation (b) from the
    # original caveat: build a fresh ChatOpenAI per invocation from a
    # fresh `_w.config.authenticate()` call.  The SDK caches the token
    # internally and only hits the IdP when refresh is due, so the
    # per-call overhead is one in-process dict lookup in the steady state
    # and one OAuth refresh roughly hourly.  `call_model` (below) calls
    # `_build_gateway_llm()` on every request when in gateway mode.
    from langchain_openai import ChatOpenAI
    from databricks.sdk import WorkspaceClient as _WorkspaceClient

    _w = _WorkspaceClient()
    _gateway_base_url = f"{_w.config.host.rstrip('/')}/ai-gateway/mlflow/v1"

    def _build_gateway_llm():
        """Construct a ChatOpenAI client pointing at the AI Gateway with the
        SDK's *current* bearer token.  Called fresh on every invocation by
        `call_model` so a rotated OAuth M2M token in the credentials
        provider is picked up immediately.  langchain-openai's ChatOpenAI
        bakes the api_key into its internal openai.OpenAI client at
        construction time and offers no callable-api_key hook, hence the
        full rebuild rather than a token-only update."""
        _auth_headers = _w.config.authenticate()
        _bearer = _auth_headers.get("Authorization", "")
        if not _bearer.lower().startswith("bearer "):
            raise RuntimeError(
                f"Databricks SDK auth headers do not contain a Bearer token "
                f"(got header keys: {list(_auth_headers.keys())}). "
                f"Cannot route refund agent through AI Gateway in this auth "
                f"mode — fall back to direct foundation-model endpoint by "
                f"unsetting AI_GATEWAY_ENDPOINT_NAME."
            )
        return ChatOpenAI(
            model=AI_GATEWAY_ENDPOINT_NAME,
            base_url=_gateway_base_url,
            api_key=_bearer[len("Bearer "):],
            max_tokens=2000,
        )

    # Initial instance: needed for `bind_tools(...)` further down so the
    # tool-bound runnable can be wired up at module load.  The instance is
    # NOT what serves requests in steady state — `call_model` rebuilds via
    # `_build_gateway_llm()` per invocation to pick up rotated tokens.
    llm = _build_gateway_llm()
else:
    # Direct foundation-model endpoint (default behaviour for `default` and
    # `complaints` targets — no gateway routing).  ChatDatabricks handles
    # auth refresh internally via the SDK on every call, so no per-call
    # rebuild is needed.
    _build_gateway_llm = None
    llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

# At endpoint startup, try to load the prompt from the MLflow Prompt
# Registry so that editing the prompt and bumping the `production` alias
# takes effect on the next endpoint replica restart — without rebuilding
# the UC model.  Falls back to the literal _FALLBACK_PROMPT below if the
# registry is empty (first-ever deploy) or unreachable (auth, etc.).
# The literal is the source of truth: the Prompt Registry cell at the
# bottom of stages/refunder_agent.ipynb extracts it via regex and
# registers it as a new version on every deploy.
PROMPT_URI = f"prompts:/{CATALOG}.prompts.refund_system@production"

_FALLBACK_PROMPT = """You are RefundGPT, a CX agent responsible for refund decisions on food delivery orders.

    You can call tools to gather the information you need. Start with an `order_id`.

    Instructions:
    1. Call `order_details(order_id)` first to get event history and confirm the id is valid and the order was delivered.
    2. Figure out the delivery duration by calling `get_order_delivery_time(order_id)`.
    3. Extract the location (either directly or from the first event's body).
    4. Call `get_location_timings(location)` to get the P50/P75/P99 values.
    5. Compare actual delivery time to those percentiles.

    Refund policy:

    A) SLA-based refund (primary path):
       - If the order arrived AFTER the P75 delivery time: recommend a `partial` or `full` refund based on how late.
       - If the order arrived BEFORE the P75: no SLA-based refund.

    B) Goodwill credit (only when complaint context is provided in the user message):
       The user may include lines such as:
           Customer complaint: "<text>"
           Complaint category: <category>
           Complaint agent suggested credit: $<amount>
       When all three are present AND the SLA path returns "none", you MAY ratify the
       complaint agent's goodwill credit:
       - Set `refund_class` = "partial"
       - Set `refund_usd` to the suggested credit amount (capped at $10)
       - In `reason`, note that the order was on time per SLA but a goodwill credit
         is being issued in response to the customer's complaint (cite the category).
       Only ratify when the suggested credit is plausible (>$0 and ≤$10) and the
       complaint category is non-empty. Otherwise return "none" with an SLA-based reason.

    When NO complaint context is provided, behave exactly as the SLA-based path (A) —
    do not invent goodwill credits.

    Output a single-line JSON with these fields:
    - `refund_usd` (float),
    - `refund_class` ("none" | "partial" | "full"),
    - `reason` (short human explanation. If goodwill, say so explicitly.)

    You must return only the JSON. No extra text or markdown."""

# Required at endpoint startup: prompt registry URI defaults vary across
# Model Serving runtimes; without this the 3-part UC name is treated as
# an opaque string in workspace MLflow and load_prompt raises NotFound.
mlflow.set_registry_uri("databricks-uc")

try:
    system_prompt = mlflow.genai.load_prompt(PROMPT_URI).template
    print(f"[refund-agent] Loaded system prompt from {PROMPT_URI}")
except Exception as _exc:
    print(
        f"[refund-agent] Failed to load {PROMPT_URI} "
        f"({type(_exc).__name__}: {_exc}); using _FALLBACK_PROMPT."
    )
    system_prompt = _FALLBACK_PROMPT

###############################################################################
## Define tools for your agent, enabling it to retrieve data or take actions
## beyond text generation.
##
## We use plain @tool-decorated wrappers (instead of UCFunctionToolkit) so
## the UC functions are only introspected/executed at REQUEST time, not at
## model LOAD time.  This is the same lazy pattern used by complaint_agent
## and support_request_agent, and is required for first-attempt success of
## agents.deploy() — see _client() above for the full rationale.
###############################################################################


@tool
def get_order_details(order_id: str) -> str:
    """Get the full event history for an order (creation, accepted, dispatched,
    delivered, etc).  Use this first to verify the order id is valid and to
    confirm the order was delivered."""
    return str(_client().execute_function(
        f"{CATALOG}.ai.get_order_details", {"oid": order_id}
    ).value)


@tool
def get_order_delivery_time(order_id: str) -> str:
    """Return the creation timestamp, delivered timestamp, and elapsed delivery
    duration for an order.  Use this to compute the actual delivery time."""
    return str(_client().execute_function(
        f"{CATALOG}.ai.get_order_delivery_time", {"oid": order_id}
    ).value)


@tool
def get_location_timings(location: str) -> str:
    """Return the P50/P75/P99 delivery time percentiles for a kitchen location
    so the actual delivery time can be compared to the SLA bands."""
    return str(_client().execute_function(
        f"{CATALOG}.ai.get_location_timings", {"loc": location}
    ).value)


tools = [get_order_details, get_order_delivery_time, get_location_timings]

#####################
## Define agent logic
#####################


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[Sequence[BaseTool], ToolNode],
    system_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    # Define the function that determines which node to go to
    def should_continue(state: ChatAgentState):
        messages = state["messages"]
        last_message = messages[-1]
        # If there are function calls, continue. else, end
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}]
            + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: ChatAgentState,
        config: RunnableConfig,
    ):
        # Gateway-mode token refresh: when routing through the AI Gateway
        # we rebuild ChatOpenAI on every call with a fresh OAuth M2M token
        # from the SDK.  Without this, warm endpoints fail with HTTP 400
        # / "Invalid Token" from the gateway once the cached token's ~1h
        # window closes.  Direct ChatDatabricks mode handles auth refresh
        # internally and reuses `model_runnable` as-is.
        if _build_gateway_llm is not None:
            fresh_model = _build_gateway_llm().bind_tools(tools)
            response = (preprocessor | fresh_model).invoke(state, config)
        else:
            response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()


class LangGraphChatAgent(ChatAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        request = {"messages": self._convert_messages_to_dict(messages)}

        result_messages = []
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                result_messages.extend(
                    ChatAgentMessage(**msg) for msg in node_data.get("messages", [])
                )
        for i in range(len(result_messages) - 1, -1, -1):
            msg = result_messages[i]
            role = msg.role if hasattr(msg, "role") else (msg.get("role") if isinstance(msg, dict) else None)
            content = msg.content if hasattr(msg, "content") else (msg.get("content", "") if isinstance(msg, dict) else "")
            if role == "assistant" and content:
                try:
                    parsed = RefundDecision.model_validate_json(content)
                    orig_id = getattr(msg, 'id', None) or str(_uuid.uuid4())
                    result_messages[i] = ChatAgentMessage(id=orig_id, role="assistant", content=parsed.model_dump_json())
                except (ValidationError, Exception):
                    pass
                break
        return ChatAgentResponse(messages=result_messages)

    def predict_stream(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> Generator[ChatAgentChunk, None, None]:
        request = {"messages": self._convert_messages_to_dict(messages)}
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                yield from (
                    ChatAgentChunk(**{"delta": msg}) for msg in node_data["messages"]
                )


# Create the agent object, and specify it as the agent object to use when
# loading the agent back for inference via mlflow.models.set_model()
agent = create_tool_calling_agent(llm, tools, system_prompt)
AGENT = LangGraphChatAgent(agent)
mlflow.models.set_model(AGENT)

In [0]:
import time

sample_order_id = None
for attempt in range(12):
    rows = spark.sql(f"""
        SELECT order_id 
        FROM {CATALOG}.lakeflow.all_events 
        WHERE event_type='delivered'
        LIMIT 1
    """).collect()
    if rows:
        sample_order_id = rows[0]['order_id']
        break
    print(f"No delivered events yet (attempt {attempt+1}/12). Waiting 30s for pipeline data...")
    time.sleep(30)

if not sample_order_id:
    raise RuntimeError(
        f"No delivered events found in {CATALOG}.lakeflow.all_events after 6 minutes. "
        "Ensure the Canonical_Data and Lakeflow pipeline stages completed and processed data."
    )

In [0]:
assert sample_order_id is not None
print(sample_order_id)

In [0]:
import mlflow
import sys
import os

# Use the absolute path captured by `%%writefilev` (see cell 13). This is
# stable across CWD drift / kernel restarts; the older `sys.path.append(os.getcwd())`
# pattern was failing with `ModuleNotFoundError: No module named 'agent'` when
# CWD changed between the writefile cell and this one.
_agent_py_path = _WRITEFILEV_ABS_PATHS.get("agent.py")
if _agent_py_path and os.path.exists(_agent_py_path):
    _agent_dir = os.path.dirname(_agent_py_path)
    print(f"Importing agent from: {_agent_py_path}")
else:
    # Defensive fallback: try the same locations the old code did, plus
    # /databricks/driver which is the typical CWD on classic clusters,
    # and the new pinned local-disk dir used by `%%writefilev`.
    _candidate_dirs = [
        os.getcwd(),
        "/databricks/driver",
        "/local_disk0/tmp/caspers_writefilev",
        "/local_disk0/tmp",
        "/tmp/caspers_writefilev",
        "/tmp",
    ]
    try:
        _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        _candidate_dirs.append(os.path.dirname(_nb_path))
    except Exception:
        pass
    _agent_dir = next((d for d in _candidate_dirs if os.path.exists(os.path.join(d, "agent.py"))), None)
    if _agent_dir is None:
        raise FileNotFoundError(
            f"agent.py not found. _WRITEFILEV_ABS_PATHS={_WRITEFILEV_ABS_PATHS}, "
            f"CWD={os.getcwd()}, candidates tried: {_candidate_dirs}"
        )
    _agent_py_path = os.path.join(_agent_dir, "agent.py")
    print(f"Importing agent from fallback dir: {_agent_py_path}")

if _agent_dir not in sys.path:
    sys.path.insert(0, _agent_dir)

# Invalidate any cached `agent` module that may be left over from a prior
# attempt on the same warehouse (e.g. after a transient failure) — without
# this, a partially-imported / stale entry in `sys.modules` can cause
# subsequent `from agent import ...` to fail or return stale symbols.
sys.modules.pop("agent", None)

from agent import LLM_ENDPOINT_NAME
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from pkg_resources import get_distribution

# Resource list for `agents.deploy()` — auto-grants permissions on each
# resource to the deployment SP at deploy time.
#
# IMPORTANT — gateway routing is special:
# When AI_GATEWAY_ENDPOINT_NAME is set, we deliberately do NOT add a
# DatabricksServingEndpoint(...) entry for it.  v2 Beta Unity AI Gateway
# endpoints live under a separate API surface (`/api/2.0/ai-gateway/*`)
# and are NOT discoverable via the regular `/api/2.0/serving-endpoints/*`
# API that `mlflow.models.resources.DatabricksServingEndpoint` validates
# against.  Listing the gateway here causes `agents.deploy()` to fail
# with "NOT_FOUND: Dependent serving endpoint <gateway> does not exist".
#
# Consequence: the deployment SP must have CAN_QUERY on the gateway via
# some other path — grant manually in the UI (AI Gateway → endpoint →
# Permissions) or via an `account users` ACL on the gateway.  At
# request time the agent authenticates with the SP's OAuth token via
# `WorkspaceClient().config.authenticate()` (see agent.py), so a CAN_QUERY
# grant is sufficient.
#
# Listing the UC functions lets agents.deploy() auto-grant EXECUTE on
# them to the endpoint SP.  Combined with the lazy @tool wrappers in
# agent.py (which defer UC introspection until request time), this lets
# model LOAD complete without needing any UC permission, so the first
# deploy succeeds even while the SP's `account users` membership is
# still propagating.
resources = [
    DatabricksFunction(function_name=f"{CATALOG}.ai.get_order_details"),
    DatabricksFunction(function_name=f"{CATALOG}.ai.get_order_delivery_time"),
    DatabricksFunction(function_name=f"{CATALOG}.ai.get_location_timings"),
]
if not AI_GATEWAY_ENDPOINT_NAME:
    # No gateway in play → the agent calls the foundation-model serving
    # endpoint directly, which IS a regular serving endpoint and CAN
    # safely be auto-granted via the resources list.
    resources.insert(0, DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME))

input_example = {
    "messages": [
        {
            "role": "user",
            "content": f"{sample_order_id}"
        }
    ]
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent_v2",
        python_model=_agent_py_path,
        input_example=input_example,
        resources=resources,
        pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"mlflow=={get_distribution('mlflow').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            # langchain-openai is only USED when AI_GATEWAY_ENDPOINT_NAME is set
            # (gateway routing on the `all` target), but we always pin it so
            # `import` succeeds at model-load time on every target — the agent
            # module imports `from langchain_openai import ChatOpenAI` lazily
            # inside the gateway branch, but a stale wheel cache or a missing
            # transitive dep can still surface at import time, and shipping
            # the same wheel set in every target keeps the failure modes
            # consistent.
            f"langchain-openai=={get_distribution('langchain-openai').version}",
            f"langgraph=={get_distribution('langgraph').version}",
        ],
    )

mlflow.set_active_model(model_id = logged_agent_info.model_id)

#### log refunder to `UC`

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateReady

mlflow.set_registry_uri("databricks-uc")

UC_MODEL_NAME = f"{CATALOG}.ai.refunder"
endpoint_name = dbutils.widgets.get("REFUND_AGENT_ENDPOINT_NAME")


def _endpoint_already_serving(name: str, uc_model_name: str) -> bool:
    """Return True iff a serving endpoint is READY and already serving uc_model_name.

    Used to short-circuit register_model + agents.deploy on re-runs of this
    stage when the endpoint from a previous run is still healthy — saves ~15
    minutes of cold container build + serving provisioning.  To force a fresh
    deploy after editing agent code, delete the endpoint and rerun the stage.
    """
    try:
        ep = WorkspaceClient().serving_endpoints.get(name)
    except Exception:
        return False
    if not ep.state or ep.state.ready != EndpointStateReady.READY:
        return False
    cfg = getattr(ep, "config", None) or getattr(ep, "pending_config", None)
    if not cfg:
        return False
    served = []
    for se in (getattr(cfg, "served_entities", None) or []):
        n = getattr(se, "entity_name", None)
        if n:
            served.append(n)
    for sm in (getattr(cfg, "served_models", None) or []):
        n = getattr(sm, "model_name", None)
        if n:
            served.append(n)
    return uc_model_name in served


_reuse_endpoint = _endpoint_already_serving(endpoint_name, UC_MODEL_NAME)

if _reuse_endpoint:
    print(
        f"\u267b\ufe0f Endpoint {endpoint_name} is already READY and serving {UC_MODEL_NAME}; "
        f"skipping register_model + agents.deploy (saves ~15 min). "
        f"Delete the endpoint to force a fresh deploy."
    )
    uc_registered_model_info = None
else:
    # register the model to UC
    uc_registered_model_info = mlflow.register_model(
        model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
    )

#### deploy the agent to model serving

In [ ]:
import mlflow

# Create prod experiment for production inference traces.
# The serving endpoint logs every request as a trace; pointing
# MLFLOW_EXPERIMENT_ID at this experiment (in the agents.deploy() call
# below) ensures those traces land somewhere stable and findable, instead
# of the auto-created /Serving/{endpoint} experiment.  Mirrors the pattern
# in stages/complaint_agent.ipynb so the runbook can link to a single
# `/Shared/{CATALOG}_refund_agent_prod` URL across deploys.
prod_experiment_name = f"/Shared/{CATALOG}_refund_agent_prod"

# set_experiment creates the experiment if it doesn't exist, or activates it if it does.
prod_experiment = mlflow.set_experiment(prod_experiment_name)
prod_experiment_id = prod_experiment.experiment_id
print(f"✅ Using prod experiment: {prod_experiment_name} (ID: {prod_experiment_id})")

# Track the experiment in uc_state so `databricks bundle run cleanup` deletes it.
import sys
sys.path.append('../utils')
from uc_state import add

experiment_data = {
    "experiment_id": prod_experiment_id,
    "name": prod_experiment_name,
}
add(CATALOG, "experiments", experiment_data)
print(f"✅ Added prod experiment to UC state")

In [0]:
from datetime import timedelta

from databricks import agents

if _reuse_endpoint:
    deployment_info = None
    print(f"\u2705 Endpoint {endpoint_name} is READY (reused from previous deploy)")
else:
    deployment_info = agents.deploy(
        model_name=UC_MODEL_NAME,
        model_version=uc_registered_model_info.version,
        scale_to_zero=False,
        endpoint_name=endpoint_name,
        environment_vars={"MLFLOW_EXPERIMENT_ID": str(prod_experiment_id)},
    )

    # Block the stage until the endpoint finishes deploying so the task
    # fails fast (and visibly) when the deploy fails, instead of returning
    # SUCCESS while the container is still building / failing in the
    # background.  Matches the wait pattern used by complaint_agent.
    workspace = WorkspaceClient()
    ready_endpoint = workspace.serving_endpoints.wait_get_serving_endpoint_not_updating(
        name=endpoint_name,
        timeout=timedelta(minutes=30),
    )

    if ready_endpoint.state.ready != EndpointStateReady.READY:
        raise RuntimeError(
            f"Endpoint {endpoint_name} is {ready_endpoint.state.ready} after deployment; retry or investigate."
        )

    print(f"\u2705 Endpoint {endpoint_name} is READY")

In [ ]:
# === Grant UC perms to the endpoint's runtime System Service Principal ===
#
# Model serving endpoints (including ones created by agents.deploy()) run
# their inference container as a workspace-level SCIM SP whose displayName
# is "System Service Principal".  These SPs are NOT members of
# `account users`, so the catalog/schema/function grants made to
# `account users` elsewhere in the bundle do NOT apply to them.
#
# Result without this cell: every fresh agent endpoint fails on its first
# tool call with PERMISSION_DENIED ("USE CATALOG" / "USE SCHEMA" / "EXECUTE")
# until somebody manually grants permissions.  The cell below discovers
# the System SPs via SCIM and grants them the perms they need.  Idempotent
# (re-granting an existing privilege in UC is a no-op).
#
# Skipped when we reused an existing endpoint (no fresh SP to grant to).
if deployment_info is not None:
    import sys
    sys.path.append('../utils')
    from agent_runtime_grants import grant_agent_runtime_perms

    # Pass endpoint_name so the helper also grants to the endpoint
    # creator — that's the actual runtime identity in EMBEDDED_CREDENTIALS
    # mode workspaces (where 'System Service Principal' isn't created and
    # `account users` may be empty at the workspace level).
    grant_agent_runtime_perms(
        spark,
        CATALOG,
        workspace_client=WorkspaceClient(),
        endpoint_name=endpoint_name,
    )
else:
    print("♻ Endpoint reused — skipping runtime SP grants (already applied on first deploy).")


In [0]:
print(deployment_info)

##### record model in state

In [ ]:
# Also add to UC-state — but only when we actually deployed a new endpoint.
# On the reuse path the endpoint was already registered by a previous run,
# so re-adding here would just create a duplicate uc_state row.
if deployment_info is not None:
    import sys
    sys.path.append('../utils')
    from uc_state import add

    add(dbutils.widgets.get("CATALOG"), "endpoints", deployment_info)
else:
    print("\u267b\ufe0f Endpoint already tracked in uc_state from a previous deploy; skipping add.")

#### Production Monitoring

Register MLflow scorers on the prod experiment so every live request that
hits the refund agent is automatically scored.  Mirrors the in-stage
pattern in `stages/complaint_agent.ipynb` and the helper in
`demos/operational-dashboard-demo/evaluation.ipynb`.

Scorer set (4, all at 100% sampling):

- `safety` — built-in `Safety()` LLM judge for harmful or inappropriate content
- `relevance_to_query` — built-in `RelevanceToQuery()` LLM judge — does the answer address the question
- `operational_quality` — generic `Guidelines` — concrete data, not a hedge
- `refund_policy_compliance` — refund-specific `Guidelines` — recommendation matches the policy

In [ ]:
from mlflow.genai.scorers import (
    Safety,
    RelevanceToQuery,
    Guidelines,
    ScorerSamplingConfig,
    list_scorers,
)


def _register_scorer(scorer_obj, name: str, sample_rate: float = 1.0):
    """Idempotent register-or-restart, scoped to the prod experiment.

    Re-running the stage hits the same code path; calling .register() on
    an already-registered scorer raises ValueError, so we look it up first
    and just .start() it instead.  Same shape as
    stages/complaint_agent.ipynb and demos/.../evaluation.ipynb.
    """
    existing = {s.name: s for s in list_scorers(experiment_id=prod_experiment_id)}
    sampling = ScorerSamplingConfig(sample_rate=sample_rate)
    if name in existing:
        existing[name].start(sampling_config=sampling)
        print(f"  ↺ {name} — restarted at {sample_rate:.0%} sample rate")
        return existing[name]
    registered = scorer_obj.register(name=name, experiment_id=prod_experiment_id)
    registered.start(sampling_config=sampling)
    print(f"  ✅ {name} — registered + started at {sample_rate:.0%} sample rate")
    return registered


# Baseline (every agent in the bundle gets these three).
_register_scorer(Safety(),           name="safety",             sample_rate=1.0)
_register_scorer(RelevanceToQuery(), name="relevance_to_query", sample_rate=1.0)
_register_scorer(
    Guidelines(
        name="operational_quality",
        guidelines=(
            "The response must include specific data points such as numbers, "
            "percentages, dates, order IDs, or named locations. "
            "The response must not be a generic hedge or refusal "
            "(e.g. 'I don't have access'). "
            "The response must directly answer the question asked."
        ),
    ),
    name="operational_quality",
    sample_rate=1.0,
)

# Domain — refund-specific policy compliance.
_register_scorer(
    Guidelines(
        name="refund_policy_compliance",
        guidelines=[
            "Recommendations must be one of: no refund, partial refund, or full refund — never anything outside this set.",
            "If a refund is recommended, the response must cite the order_id it applies to.",
            "If the order is older than the refund window, the response must explicitly mention that the refund window has expired.",
            "The response must reference the specific reason from the order data (e.g. late delivery, missing item, food quality).",
        ],
    ),
    name="refund_policy_compliance",
    sample_rate=1.0,
)

print("✅ Production monitoring enabled — 4 scorers active at 100% sampling")

#### Prompt Registry

Register the system prompt that was just baked into `agent.py` under
`{CATALOG}.prompts.refund_system` so it lives alongside the deployed model
in Unity Catalog. We extract the template directly from `agent.py` rather
than re-declaring it here, so the registry tracks **exactly** what was
deployed (no drift possible).

Re-running the stage creates a new prompt version and bumps the
`production` alias. The schema and all prompts are dropped automatically
by `DROP CATALOG ... CASCADE` in `destroy.ipynb`, so no extra cleanup
wiring is needed.

In [ ]:
import re
import sys

sys.path.append('../utils')
from prompt_registry import seed_prompt_history

# Use the absolute path captured by %%writefilev (see cell 13) so this works
# even when CWD has drifted between cells.
_agent_py_path = _WRITEFILEV_ABS_PATHS.get("agent.py", "agent.py")
with open(_agent_py_path) as f:
    _agent_py = f.read()

# Extract from _FALLBACK_PROMPT — the source-of-truth literal in agent.py.
# The deployed agent prefers mlflow.genai.load_prompt() at startup but
# falls back to this literal if the registry is empty or unreachable.
_match = re.search(r'_FALLBACK_PROMPT\s*=\s*"""(.*?)"""', _agent_py, re.DOTALL)
if not _match:
    raise RuntimeError(
        "Could not extract _FALLBACK_PROMPT from agent.py. "
        "If the prompt block was renamed, update this regex."
    )

_uc_version = (
    uc_registered_model_info.version
    if uc_registered_model_info is not None
    else "reused-endpoint"
)

# Two earlier versions of the refund prompt, seeded on first deploy so the
# Prompt Registry UI shows v1 → v2 → v3 history. These are demo seeds, NOT a
# real engineering changelog — seed_prompt_history tags each with
# is_demo_seed="true" so anyone auditing the registry can tell.
_REFUND_V1 = (
    "You are a refund agent for a food delivery service. "
    "Given an order_id, decide whether to issue a refund and how much. "
    "Return a single-line JSON with `refund_usd` (float), `refund_class` "
    "(\"none\" | \"partial\" | \"full\"), and `reason` (short explanation)."
)
_REFUND_V2 = """You are RefundGPT, a CX agent responsible for refund decisions on food delivery orders.

    You can call tools to gather the information you need. Start with an `order_id`.

    Instructions:
    1. Call `order_details(order_id)` first to get event history and confirm the id is valid and the order was delivered.
    2. Figure out the delivery duration by calling `get_order_delivery_time(order_id)`.
    3. Extract the location (either directly or from the first event's body).
    4. Call `get_location_timings(location)` to get the P50/P75/P99 values.
    5. Compare actual delivery time to those percentiles.

    Refund policy (SLA-based):
       - If the order arrived AFTER the P75 delivery time: recommend a `partial` or `full` refund based on how late.
       - If the order arrived BEFORE the P75: no refund.

    Output a single-line JSON with these fields:
    - `refund_usd` (float),
    - `refund_class` (\"none\" | \"partial\" | \"full\"),
    - `reason` (short human explanation).

    You must return only the JSON. No extra text or markdown."""

_common_tags = {
    "agent": "refund",
    "stage": "refunder_agent",
    "uc_model": UC_MODEL_NAME,
    "consumed_via": "mlflow.genai.load_prompt at endpoint startup",
}

seed_prompt_history(
    spark=spark,
    catalog=CATALOG,
    name="refund_system",
    historical=[
        {
            "template": _REFUND_V1,
            "commit_message": "v1: bare-bones refund decisioner, no SLA logic or tool use (demo history seed)",
            "tags": _common_tags,
        },
        {
            "template": _REFUND_V2,
            "commit_message": "v2: added tool-calling + SLA-based refund policy (P75 cutoff) (demo history seed)",
            "tags": _common_tags,
        },
    ],
    current={
        "template": _match.group(1).strip(),
        "commit_message": f"v3 (production): SLA + goodwill credit path — UC model version {_uc_version}",
        "tags": {**_common_tags, "uc_model_version": str(_uc_version)},
    },
)